In [0]:
%sql
-- =============================================================================
-- ETL Silver: Ingesta limpia y tipada desde Bronze -> Silver
-- Granularidad destino: 1 fila por (video_id, hashtag)
-- Deduplicacion por PK natural usando QUALIFY ROW_NUMBER()
-- =============================================================================
MERGE INTO tiktok_data_eng.silver.silver_tiktok AS target
USING (
    SELECT
        search_type,
        search_hashtag,
        video_id,
        NULLIF(description, 'None')           AS description,
        to_timestamp(created_at)              AS created_at,
        video_url,
        NULLIF(region, 'None')                AS region,
        CAST(duration AS INT)                 AS duration,
        CAST(video_width AS INT)              AS video_width,
        CAST(video_height AS INT)             AS video_height,
        ratio,
        CAST(is_ad AS BOOLEAN)                AS is_ad,
        CAST(is_photo AS BOOLEAN)             AS is_photo,
        CAST(is_paid_content AS BOOLEAN)      AS is_paid_content,
        description_language,
        CAST(plays AS BIGINT)                 AS plays,
        CAST(likes AS BIGINT)                 AS likes,
        CAST(comments AS BIGINT)              AS comments,
        CAST(shares AS BIGINT)                AS shares,
        CAST(saves AS BIGINT)                 AS saves,
        author_id,
        author_unique_id,
        author_nickname,
        CAST(author_verified AS BOOLEAN)      AS author_verified,
        NULLIF(author_signature, 'None')      AS author_signature,
        hashtags_explodido                    AS hashtag,
        fecha_ingesta
    FROM tiktok_data_eng.bronze.tiktok_bronze
    LATERAL VIEW EXPLODE(
        from_json(regexp_replace(hashtags, "'", '"'), 'array<string>')
    ) AS hashtags_explodido
    WHERE hashtags_explodido IS NOT NULL
      AND hashtags_explodido != 'None'
      AND hashtags_explodido != ''
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY video_id, hashtags_explodido 
        ORDER BY fecha_ingesta DESC
    ) = 1
) AS source
    ON target.video_id = source.video_id
   AND target.hashtag  = source.hashtag
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Verificacion de calidad post-carga
SELECT 
    count(*) as total_filas,
    count(distinct video_id) as videos_unicos,
    count(distinct hashtag) as hashtags_unicos,
    count(if(video_id is null, 1, null)) as nulos_video_id,
    count(if(hashtag is null, 1, null)) as nulos_hashtag,
    count(*) - count(distinct concat(video_id, '__', hashtag)) as duplicados_pk
FROM tiktok_data_eng.silver.silver_tiktok;